# Neural Privilege Separation — Experiment 018
## Universal Safety Representation: Cross-Benchmark Generalization of Activation-Space Refusal Probes

**Builds directly on Exp017** (Qwen2.5-3B-Instruct, probe-direction ensemble firewall). Exp017's
architecture — manifest-based resume, per-artifact Drive checkpointing, residual-stream
extraction, per-layer logistic-regression probe directions, ROC-based calibration — is kept
unchanged.

**Redesign rationale (this revision).** The original design used leave-one-benchmark-out (LOBO)
as the primary evaluation. LOBO is fundamentally ill-posed for this benchmark set: XSTest is the
only benchmark that contains both compliant and refusal-worthy examples — HarmBench,
JailbreakBench, SorryBench, and StrongREJECT are refusal-only. A LOBO fold that holds out any of
those four trains on a benign-example pool drawn only from the remaining benchmarks, which is a
different (and weaker) question than "does an unseen benchmark generalize." **LOBO is kept in
Section 11 as a secondary diagnostic** (it still produces a real, interpretable number for the
XSTest fold), but it is no longer this notebook's central evidence.

**The actual scientific hypothesis under test is:**

> *Do multiple safety benchmarks share a benchmark-independent activation-space representation
> of unsafe intent?*

That hypothesis is now tested directly via a **complete cross-benchmark transfer matrix**
(Section 12): every specified train-configuration is evaluated against every benchmark's own
held-out test split, with the full metric set (precision / recall / F1 / FPR / AUROC / balanced
accuracy) reported per cell — not just recall. Three representation-geometry analyses (probe
similarity across training combinations, PCA/UMAP/hierarchical-clustering of probe directions,
and stability across layers/seeds) characterize *how* probes relate to each other, and two
negative controls (label permutation, random-direction baseline) establish whether any of it is
statistically meaningful rather than an artifact of high-dimensional geometry.

**How to read the transfer matrix:** rows are training configurations, columns are benchmarks
evaluated on their own untouched test split. A cell is a genuine held-out number even when the
row's training combo includes that column's benchmark, because each benchmark's test rows were
carved out (Section 8) before any training happened and never re-enter any training pool. High
values across a column regardless of row indicate that column's phenomenon is easy to detect
from any training mix; high values across a row regardless of column indicate that row's
training mix produces a broadly transferable direction — this second pattern is the one
"universal representation" would predict.

**What the permutation control validates:** if AUROC computed after independently shuffling
training labels is indistinguishable from AUROC on real labels, the "signal" a probe finds is
not actually tied to the safe/refusal distinction — it's exploiting some other structure (e.g.
benchmark-of-origin) or is noise. A large, consistent gap between real and shuffled AUROC is
necessary (not sufficient) evidence that the learned direction is tracking the intended concept.

**What the random-direction baseline validates:** in a high-dimensional residual stream, two
independently drawn random vectors already have non-zero expected cosine similarity due to
concentration-of-measure effects, and that expected similarity shrinks as dimensionality grows.
Comparing real probe-pair cosine similarities against a random-vector-pair baseline of the same
dimensionality answers "is this convergence bigger than the geometry would produce by chance,"
rather than taking any positive cosine similarity as evidence of shared representation on its
own.

**Design choices carried over:** activation extraction remains the only step expensive enough to
need GPU time and Drive-checkpointed resume (one extraction per benchmark per split — 15 total
for 5 benchmarks × train/calibration/test). Every probe subsequently trained on those cached
activations (LOBO folds, transfer-matrix cells, layer/seed stability, permutation runs) is a fast
CPU-only logistic regression fit; these are individually cached via manifest stage keys so a
disconnect mid-run resumes from the last completed cell/combination rather than restarting, per
the resumability requirement, even though their absolute cost is negligible.

Sections:
1. Environment Setup
2. Install Dependencies
3. Mount Google Drive
4. Configuration
5. Resume Logic
6. Load Model
7. Benchmark Adapters — Load & Canonicalize XSTest / JailbreakBench / HarmBench / SorryBench / StrongREJECT
8. Per-Benchmark Stratified Splits
9. Hidden-State Extraction (per benchmark, per split — cached & resumable)
9b. Prompt Length Report (per-benchmark token statistics, computed pre-extraction)
10. Probe Training & Calibration Utilities (unchanged mechanism from Exp017)
11. Leave-One-Benchmark-Out (LOBO) Evaluation — secondary diagnostic
12. Cross-Benchmark Transfer Matrix — primary analysis
13. Probe Similarity Analysis
13b. Probe Geometry Analysis (PCA / UMAP / Hierarchical Clustering of probe directions)
14. Layer Stability Analysis
14b. Permutation Control (negative control)
14c. Random Direction Baseline (negative control)
15. Seed Stability Analysis
16. PCA / UMAP Representation Visualization (activations)
17. Save Outputs
18. Final Results Report


## 1. Environment Setup

In [ ]:
import os, sys, json, time, gc, pickle, random, traceback, subprocess, warnings, hashlib
from dataclasses import dataclass, field, asdict
from pathlib import Path
from datetime import datetime, timedelta
from itertools import product, combinations

warnings.filterwarnings("ignore")

SEED = 42

def set_all_seeds(seed=SEED):
    import numpy as np
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass

random.seed(SEED)
print(f"[Exp018] Environment setup started at {datetime.now().isoformat()}")

## 2. Install Dependencies

In [ ]:
REQUIRED_PACKAGES = [
    "transformers>=4.42.0",
    "accelerate",
    "sentencepiece",
    "scikit-learn",
    "scipy",
    "matplotlib",
    "pandas",
    "tqdm",
    "datasets",
]
OPTIONAL_PACKAGES = ["umap-learn"]  # only used if available; PCA viz never depends on it

def pip_install(pkg):
    try:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
        return True
    except subprocess.CalledProcessError as e:
        print(f"  [WARN] Failed to install {pkg}: {e}")
        return False

print("[Exp018] Installing required dependencies (per-package, fault-tolerant)...")
for pkg in REQUIRED_PACKAGES:
    ok = pip_install(pkg)
    print(f"  {'OK' if ok else 'FAILED'}: {pkg}")

print("[Exp018] Installing optional dependencies (failure here is fine, features degrade gracefully)...")
for pkg in OPTIONAL_PACKAGES:
    ok = pip_install(pkg)
    print(f"  {'OK' if ok else 'SKIPPED'}: {pkg}")

try:
    import umap  # noqa: F401
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
print(f"[Exp018] umap-learn available: {UMAP_AVAILABLE}")

import torch
import numpy as np
import pandas as pd

set_all_seeds(SEED)
print(f"[Exp018] torch={torch.__version__} cuda_available={torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"[Exp018] GPU: {torch.cuda.get_device_name(0)}")

## 3. Mount Google Drive

In [ ]:
IN_COLAB = "google.colab" in sys.modules
DRIVE_ROOT = Path("/content/drive/MyDrive/NPS/Exp018")
USE_LOCAL_STORAGE = False

if IN_COLAB:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        if not Path("/content/drive/MyDrive").exists():
            raise RuntimeError("Drive mount reported success but MyDrive path not found")
    except Exception as e:
        print(f"[Exp018][WARN] Google Drive mount failed/halted ({e}). Falling back to LOCAL "
              f"Colab runtime storage at /content/NPS/Exp018.")
        print("[Exp018][WARN] Local runtime storage is EPHEMERAL — wiped on disconnect/reset/"
              "timeout. Download outputs (Section 17) before ending the session if Drive stays "
              "unavailable, since nothing here will persist to a future session otherwise.")
        DRIVE_ROOT = Path("/content/NPS/Exp018")
        USE_LOCAL_STORAGE = True
else:
    DRIVE_ROOT = Path("./NPS_Exp018_local")
    print("[Exp018] Not running in Colab — using local fallback directory for Drive root.")
    USE_LOCAL_STORAGE = True

DIRS = {
    "root": DRIVE_ROOT,
    "datasets": DRIVE_ROOT / "datasets",
    "activations": DRIVE_ROOT / "activations",
    "probes": DRIVE_ROOT / "probes",
    "lobo": DRIVE_ROOT / "lobo",
    "transfer": DRIVE_ROOT / "transfer",       # NEW: cross-benchmark transfer matrix (Change 1)
    "geometry": DRIVE_ROOT / "geometry",       # NEW: probe PCA/UMAP/dendrogram artifacts (Change 4)
    "controls": DRIVE_ROOT / "controls",       # NEW: permutation + random-direction baselines (Changes 5-6)
    "matrix": DRIVE_ROOT / "matrix",           # kept for backward compatibility with older runs
    "similarity": DRIVE_ROOT / "similarity",
    "figures": DRIVE_ROOT / "figures",
    "logs": DRIVE_ROOT / "logs",
}
for name, d in DIRS.items():
    d.mkdir(parents=True, exist_ok=True)

print(f"[Exp018] Drive root: {DRIVE_ROOT}")
for name, d in DIRS.items():
    print(f"  {name:12s} -> {d}")


## 4. Configuration

In [ ]:
@dataclass
class Exp018Config:
    # Model — identical to Exp015/016/017
    model_name: str = "Qwen/Qwen2.5-3B-Instruct"
    dtype: str = "bfloat16"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"
    layers: tuple = (19, 20, 21, 22)
    pooling_strategy: str = "last_token"

    # --- Benchmarks ---
    # Every benchmark this notebook knows how to load. Set to a subset to exclude one (e.g. if
    # its HF dataset ID is stale and you don't want a placeholder standing in for it).
    benchmarks_enabled: tuple = ("xstest", "jailbreakbench", "harmbench", "sorrybench", "strongreject")

    # Per-benchmark stratified split ratios (train / calibration / test), same fixed stratified
    # logic as Exp017 — each benchmark's own held-out test slice is what's used both as its
    # column in the transfer matrix and as the LOBO test set when it's the one held out.
    split_ratios: dict = field(default_factory=lambda: {"train": 0.5, "calibration": 0.2, "test": 0.3})

    # --- Prompt length normalization (Change 2) ---
    # Some SorryBench prompts exceed 3500 tokens, which caused GPU OOM during batched extraction.
    # This is the single source of truth for tokenizer truncation length; the pre-extraction
    # length report (Section 9b) also uses it to flag how many prompts per benchmark actually
    # get truncated at this value.
    max_prompt_tokens: int = 512

    # Cap examples pulled per benchmark, applied AFTER stratified label-balanced sampling
    # (Change 3), and BEFORE train/calibration/test splitting. Keeps a Colab Free GPU run
    # tractable regardless of how large the source benchmark is — activation extraction is the
    # only GPU-bound step and its cost scales with this number.
    max_examples_per_benchmark: int = 500

    # --- Leave-one-benchmark-out (secondary diagnostic — see Section 11 markdown) ---
    do_lobo: bool = True

    # --- Cross-benchmark transfer matrix (Change 1 — primary analysis, Section 12) ---
    # None = auto-build the specified configuration set: [xstest], [xstest+each other benchmark
    # individually], [all benchmarks]. Override with an explicit tuple-of-tuples to customize.
    transfer_train_combos: tuple = None

    # --- Calibration / ensemble (unchanged mechanism from Exp017) ---
    target_fpr: float = 0.02
    vote_k: int = 2
    layer_weights: dict = None

    # --- Probe geometry analysis (Change 4) ---
    # Which layer's probe directions to run PCA/UMAP/hierarchical-clustering over. None = middle
    # of CONFIG.layers (matches the activation-PCA layer choice in Section 16 for consistency).
    probe_geometry_layer: int = None

    # --- Permutation control (Change 5) ---
    n_permutations: int = 20

    # --- Seed stability ---
    # sklearn's default 'lbfgs' solver is a deterministic convex optimizer — refitting the
    # SAME data at different `random_state` values produces identical probes and would make
    # this section vacuous (cosine similarity trivially 1.0 always). Seed stability is
    # therefore measured via bootstrap resampling of the training pool at each seed, which
    # answers the real question: does the learned direction depend on which examples you
    # happened to draw, or does it converge regardless?
    stability_seeds: tuple = (1, 42, 123)

    n_bootstrap: int = 2000
    seed: int = 42

CONFIG = Exp018Config()
print(json.dumps(asdict(CONFIG), indent=2, default=str))

config_path = DIRS["root"] / "exp018_config.json"
with open(config_path, "w") as f:
    json.dump(asdict(CONFIG), f, indent=2, default=str)
print(f"[Exp018] Config saved to {config_path}")


## 5. Resume Logic\n\nSame manifest pattern as Exp017, unchanged.

In [ ]:
MANIFEST_PATH = DIRS["root"] / "manifest.json"

def load_manifest():
    if MANIFEST_PATH.exists():
        try:
            with open(MANIFEST_PATH) as f:
                m = json.load(f)
            print(f"[Resume] Loaded manifest with {len(m.get('completed_stages', []))} completed stages.")
            return m
        except (json.JSONDecodeError, OSError) as e:
            print(f"[Resume] Manifest unreadable ({e}); backing up and starting fresh.")
            MANIFEST_PATH.rename(MANIFEST_PATH.with_suffix(".corrupt.json"))
    return {"completed_stages": [], "dataset_fingerprints": {}, "last_updated": None}

def save_manifest(manifest):
    manifest["last_updated"] = datetime.now().isoformat()
    tmp = MANIFEST_PATH.with_suffix(".tmp")
    with open(tmp, "w") as f:
        json.dump(manifest, f, indent=2)
    tmp.replace(MANIFEST_PATH)

def mark_stage_done(manifest, stage_name):
    if stage_name not in manifest["completed_stages"]:
        manifest["completed_stages"].append(stage_name)
    save_manifest(manifest)

def stage_done(manifest, stage_name):
    return stage_name in manifest["completed_stages"]

MANIFEST = load_manifest()
MANIFEST.setdefault("dataset_fingerprints", {})

## 6. Load Model

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

DTYPE_MAP = {"bfloat16": torch.bfloat16, "float16": torch.float16, "float32": torch.float32}

def get_decoder_layers(model):
    return model.model.layers

def load_model_and_tokenizer(config: Exp018Config):
    print(f"[Model] Loading {config.model_name} ...")
    tokenizer = AutoTokenizer.from_pretrained(config.model_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        config.model_name,
        torch_dtype=DTYPE_MAP.get(config.dtype, torch.bfloat16),
        device_map=config.device,
    )
    model.eval()
    n_layers = model.config.num_hidden_layers
    print(f"[Model] Loaded. num_hidden_layers={n_layers}, hidden_size={model.config.hidden_size}")
    for L in config.layers:
        assert 0 <= L < n_layers, f"Layer {L} out of range for a {n_layers}-layer model"
    return model, tokenizer

MODEL, TOKENIZER = load_model_and_tokenizer(CONFIG)
mark_stage_done(MANIFEST, "model_loaded_check")

import transformers as _tf
REPRO = {
    "model_name": CONFIG.model_name,
    "transformers_version": _tf.__version__,
    "torch_version": torch.__version__,
    "seed": CONFIG.seed,
    "timestamp": datetime.now().isoformat(),
}
with open(DIRS["root"] / "reproducibility.json", "w") as f:
    json.dump(REPRO, f, indent=2)
print(json.dumps(REPRO, indent=2))

## 7. Benchmark Adapters — Load & Canonicalize

Converts each benchmark into the canonical schema: `prompt, label, category, benchmark,
expected_refusal` (`label` in `{safe, refusal}`, `expected_refusal` in `{0,1}`).

**Honesty note:** HuggingFace dataset IDs and column schemas for these benchmarks can change,
and this notebook cannot verify them live at build time. Each adapter tries a short list of
plausible dataset IDs, then detects columns defensively by searching common name variants
(`prompt`/`goal`/`behavior`/`question` for the prompt text, `label`/`is_safe`/`category` for
labeling, etc.). If every candidate fails to load — wrong ID, network unavailable, gated
dataset — it falls back to a small clearly-labeled placeholder for that benchmark and prints
a loud warning, so the notebook always completes end to end, but you should verify each
benchmark actually loaded for real before trusting its numbers (check `benchmark_load_status`
in the printed output and in `exp018_summary.json`).

In [ ]:
!pip -q install huggingface_hub
from huggingface_hub import login
import os

# SECURITY NOTE: never hardcode an HF token in a notebook cell — anyone who opens/shares this
# file (or a Colab copy of it) gets your token. Prefer, in order:
#   1) an HF_TOKEN already set in the runtime environment / Colab secrets, or
#   2) an interactive, non-echoing prompt (getpass) that is never written to disk or output.
_hf_token = os.environ.get("HF_TOKEN")
if not _hf_token:
    try:
        # Colab's userdata secrets store, if this notebook is running in Colab and the user
        # has added an HF_TOKEN secret via the key icon in the left sidebar.
        from google.colab import userdata
        _hf_token = userdata.get("HF_TOKEN")
    except Exception:
        _hf_token = None
if not _hf_token:
    import getpass
    _hf_token = getpass.getpass("Enter your Hugging Face token (input hidden; leave blank to skip login): ")

if _hf_token:
    os.environ["HF_TOKEN"] = _hf_token
    login(token=_hf_token)
    print("[Exp018] Hugging Face login complete.")
else:
    print("[Exp018][WARN] No HF token provided — dataset loads that require auth (e.g. gated "
          "repos) will fail and fall back to placeholders. Public datasets are unaffected.")


In [ ]:
"""
Exp018 Research-Grade Benchmark Loader (template)

This module is designed for NPS experiments.

Principles
----------
- No placeholder datasets.
- Fail fast.
- One adapter per benchmark.
- Benchmark-specific schema handling.
- Deterministic validation.
- Reproducible sampling.
- Easy extension.

NOTE:
Public dataset identifiers change over time. Populate DATASET_REGISTRY with
verified dataset IDs before use.
"""

from dataclasses import dataclass
from typing import Callable, Dict, List, Optional, Tuple
import pandas as pd
import numpy as np
from datasets import load_dataset

REQUIRED_COLUMNS = [
    "prompt","label","expected_refusal","label_bin","benchmark","category"
]

@dataclass
class DatasetSpec:
    candidates: List[Tuple[str, Optional[str]]]
    adapter: Callable[[pd.DataFrame], pd.DataFrame]

def _ensure(df,name):
    miss=[c for c in REQUIRED_COLUMNS if c not in df.columns]
    if miss:
        raise RuntimeError(f"{name}: missing {miss}")
    if df["prompt"].isna().any():
        raise RuntimeError(f"{name}: missing prompts")
    if df["label_bin"].isna().any():
        raise RuntimeError(f"{name}: missing labels")
    print(f"\n{name}")
    print(df["label"].value_counts())
    return df

def adapt_xstest(raw):
    out=pd.DataFrame()
    out["prompt"]=raw["prompt"]
    def f(x):
        s=str(x).lower()
        if "compliance" in s: return "compliance"
        if "refusal" in s or "reject" in s: return "refusal"
        raise ValueError(x)
    out["label"]=raw["final_label"].map(f)
    out["expected_refusal"]=(out.label=="refusal").astype(int)
    out["label_bin"]=out.expected_refusal
    out["benchmark"]="xstest"
    out["category"]=raw["type"] if "type" in raw else "unspecified"
    return _ensure(out,"xstest")

def adapt_jbb(raw):
    if "Goal" not in raw:
        raise RuntimeError("Expected Goal column")
    out=pd.DataFrame({
        "prompt":raw["Goal"],
        "label":"refusal",
        "expected_refusal":1,
        "label_bin":1,
        "benchmark":"jailbreakbench",
        "category":raw["Category"] if "Category" in raw else "behavior"
    })
    return _ensure(out,"jailbreakbench")

def adapt_stub(name):
    def inner(raw):
        raise NotImplementedError(
            f"{name} adapter must be implemented after inspecting the verified dataset schema."
        )
    return inner

def adapt_harmbench(raw):
    required = ["prompt", "category"]

    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise RuntimeError(
            f"HarmBench missing columns: {missing}\n"
            f"Available columns: {list(raw.columns)}"
        )

    out = pd.DataFrame({
        "prompt": raw["prompt"].astype(str).str.strip(),
        "label": "refusal",
        "expected_refusal": 1,
        "label_bin": 1,
        "benchmark": "harmbench",
        "category": raw["category"].fillna("unknown").astype(str)
    })

    # Drop empty prompts
    out = out[out["prompt"] != ""].reset_index(drop=True)

    return _ensure(out, "harmbench")

def adapt_sorrybench(raw):
    required = ["turns", "category"]

    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise RuntimeError(
            f"SorryBench missing columns: {missing}\n"
            f"Available columns: {list(raw.columns)}"
        )

    out = pd.DataFrame()

    out["prompt"] = raw["turns"].apply(lambda x: str(x[0]).strip() if len(x) else "")
    out["label"] = "refusal"
    out["expected_refusal"] = 1
    out["label_bin"] = 1
    out["benchmark"] = "sorrybench"
    out["category"] = raw["category"].fillna("unknown").astype(str)

    out = out[out["prompt"] != ""].reset_index(drop=True)

    return _ensure(out, "sorrybench")


def adapt_strongreject(raw):
    required = ["prompt", "category", "prompt_label"]
    missing = [c for c in required if c not in raw.columns]
    if missing:
        raise RuntimeError(
            f"StrongREJECT missing columns: {missing}\n"
            f"Available columns: {list(raw.columns)}"
        )
    out = pd.DataFrame()
    out["prompt"] = raw["prompt"].astype(str).str.strip()
    # Map labels
    label_map = {
        "unsafe": "refusal",
        "safe": "compliance"
    }

    labels = raw["prompt_label"].astype(str).str.lower().map(label_map)
    if labels.isna().any():
        unknown = sorted(raw.loc[labels.isna(), "prompt_label"].unique())
        raise RuntimeError(
            f"Unknown StrongREJECT labels: {unknown}"
        )
    out["label"] = labels
    out["expected_refusal"] = (labels == "refusal").astype(int)
    out["label_bin"] = out["expected_refusal"]
    out["benchmark"] = "strongreject"
    out["category"] = raw["category"].fillna("unknown").astype(str)

    # Remove empty prompts
    out = out[out["prompt"] != ""].reset_index(drop=True)
    return _ensure(out, "strongreject")

DATASET_REGISTRY: Dict[str,DatasetSpec]={
    "xstest":DatasetSpec(
        [("natolambert/xstest-v2-copy",None)],
        adapt_xstest),
    "jailbreakbench":DatasetSpec(
        [("JailbreakBench/JBB-Behaviors","behaviors")],
        adapt_jbb),
    "harmbench":DatasetSpec(
        [("walledai/HarmBench", "contextual")]
        ,adapt_harmbench),
    "sorrybench":DatasetSpec(
        [("sorry-bench/sorry-bench-202503",None)]
        ,adapt_sorrybench),
    "strongreject":DatasetSpec(
        [("Machlovi/strongreject-dataset",None)]
        ,adapt_strongreject),
}

def load_first(candidates):
    last=None
    for ds,cfg in candidates:
        try:
            obj=load_dataset(ds,cfg) if cfg else load_dataset(ds)
            split="train" if "train" in obj else list(obj.keys())[0]
            return obj[split].to_pandas(), ds
        except Exception as e:
            last=e
    raise RuntimeError(last)

def load_all():
    out={}
    for name,spec in DATASET_REGISTRY.items():
        if not spec.candidates:
            raise RuntimeError(f"{name}: no verified dataset configured.")
        raw,src=load_first(spec.candidates)
        print(f"Loaded {name} from {src}")
        out[name]=spec.adapter(raw)
    return out

if __name__=="__main__":
    dfs=load_all()
    print("\\nLoaded:",list(dfs))


In [ ]:
BENCHMARK_DFS = load_all()

print("\n[Exp018] Loaded benchmarks:", list(BENCHMARK_DFS.keys()))
for name, df in BENCHMARK_DFS.items():
    print(f"  {name:15s} n={len(df):5d}  label_counts={df['label'].value_counts().to_dict()}")

def sample_balanced(df, n, seed):
    """Random-sample down to at most n rows, preserving class balance where possible
    (Change 3). Each label group is sampled proportionally to its share of the group's own
    size; if a group is smaller than its proportional share, ALL of that group's rows are kept
    and the shortfall is redistributed to the other group(s) rather than silently dropped.
    Runs BEFORE train/calibration/test splitting, per the requirement."""
    if len(df) <= n:
        return df.reset_index(drop=True)
    groups = {lbl: g for lbl, g in df.groupby("label")}
    sizes = {lbl: len(g) for lbl, g in groups.items()}
    total = sum(sizes.values())
    # initial proportional targets
    targets = {lbl: int(round(n * sizes[lbl] / total)) for lbl in groups}
    # cap at each group's actual size, track shortfall to redistribute
    shortfall = 0
    for lbl in list(targets):
        if targets[lbl] > sizes[lbl]:
            shortfall += targets[lbl] - sizes[lbl]
            targets[lbl] = sizes[lbl]
    if shortfall > 0:
        # give the shortfall to groups that still have spare capacity, largest-spare first
        spare = {lbl: sizes[lbl] - targets[lbl] for lbl in targets}
        for lbl in sorted(spare, key=lambda k: -spare[k]):
            if shortfall <= 0:
                break
            take = min(shortfall, spare[lbl])
            targets[lbl] += take
            shortfall -= take
    parts = [groups[lbl].sample(min(targets[lbl], sizes[lbl]), random_state=seed) for lbl in groups]
    return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)

print(f"\n[Exp018] Balancing/capping each benchmark to max_examples_per_benchmark="
      f"{CONFIG.max_examples_per_benchmark} (class-balance preserved where possible)...")
for name in BENCHMARK_DFS:
    before = len(BENCHMARK_DFS[name])
    BENCHMARK_DFS[name] = sample_balanced(BENCHMARK_DFS[name], CONFIG.max_examples_per_benchmark, CONFIG.seed)
    after = len(BENCHMARK_DFS[name])
    tag = "unchanged" if before == after else f"{before} -> {after}"
    print(f"  {name:15s} {tag}  label_counts={BENCHMARK_DFS[name]['label'].value_counts().to_dict()}")


## 8. Per-Benchmark Stratified Splits

Each benchmark is split independently into train/calibration/test, stratified by label (fixed
per the Exp017 stratification bug — a non-stratified split can, on an unlucky shuffle,
leave a split with a single class and crash probe fitting). A benchmark's own `test` slice is
what gets used both as its column in the generalization matrix and as the LOBO test set on the
fold where it's the one held out — its `train`+`calibration` rows are what get folded into
other benchmarks' training pools, never leaking its own test rows into any training pool.

In [ ]:
def make_splits(df, ratios, seed):
    """Stratified by label: splits each class proportionally so every split gets both
    classes as long as the source data has at least a few of each."""
    df = df.copy()
    df["label_bin"] = (df["label"] == "refusal").astype(int)
    train_parts, cal_parts, test_parts = [], [], []
    for label_val, group in df.groupby("label_bin"):
        g = group.sample(frac=1.0, random_state=seed).reset_index(drop=True)
        n = len(g)
        n_train = max(1, int(n * ratios["train"])) if n >= 1 else 0
        n_cal = max(1, int(n * ratios["calibration"])) if n - n_train >= 1 else 0
        n_train, n_cal = min(n_train, n), min(n_cal, n - n_train)
        train_parts.append(g.iloc[:n_train])
        cal_parts.append(g.iloc[n_train:n_train + n_cal])
        test_parts.append(g.iloc[n_train + n_cal:])
    def _combine(parts):
        return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=seed).reset_index(drop=True)
    train, cal, test = _combine(train_parts), _combine(cal_parts), _combine(test_parts)
    for split_name, d in [("train", train), ("calibration", cal), ("test", test)]:
        n_classes = d["label_bin"].nunique() if len(d) else 0
        if n_classes < 2:
            print(f"    [Splits][WARN] '{split_name}' has only {n_classes} class(es) present.")
    return train, cal, test

BENCH_SPLITS = {}
for name, df in BENCHMARK_DFS.items():
    train, cal, test = make_splits(df, CONFIG.split_ratios, CONFIG.seed)
    BENCH_SPLITS[name] = {"train": train, "calibration": cal, "test": test}
    print(f"[Splits] {name}: train={len(train)} calibration={len(cal)} test={len(test)}")

## 9. Hidden-State Extraction

The only GPU-bound, expensive step — one extraction per benchmark per split (15 total),
individually cached and individually resumable, with a per-benchmark dataset fingerprint so a
benchmark whose loaded prompts change (e.g. a placeholder gets replaced by a real load next
session) invalidates only ITS OWN cached activations, not the other four benchmarks'.

### 9b. Prompt Length Report

Computed **before** extraction (per the requirement) so a run's console/log makes clear, up
front, how aggressive `CONFIG.max_prompt_tokens` truncation will be for each benchmark —
SorryBench in particular has prompts exceeding 3500 tokens, which is what caused GPU OOM in
earlier revisions. `pct_truncated` below is the share of each benchmark's prompts whose raw
token count already exceeds `CONFIG.max_prompt_tokens`, i.e. how much of that benchmark loses
content to truncation at extraction time.

In [ ]:
PROMPT_LENGTH_CSV_PATH = DIRS["logs"] / "prompt_length_report.csv"

def prompt_length_report(dfs, tokenizer, max_tokens):
    rows = []
    for name, df in dfs.items():
        lengths = [len(tokenizer.encode(p, add_special_tokens=False)) for p in df["prompt"]]
        lengths = np.array(lengths)
        rows.append({
            "benchmark": name,
            "n_prompts": len(lengths),
            "mean_tokens": float(lengths.mean()),
            "median_tokens": float(np.percentile(lengths, 50)),
            "p95_tokens": float(np.percentile(lengths, 95)),
            "max_tokens": int(lengths.max()),
            "max_prompt_tokens_config": max_tokens,
            "pct_truncated": float((lengths > max_tokens).mean() * 100),
        })
    return pd.DataFrame(rows)

PROMPT_LENGTH_DF = prompt_length_report(BENCHMARK_DFS, TOKENIZER, CONFIG.max_prompt_tokens)
PROMPT_LENGTH_DF.to_csv(PROMPT_LENGTH_CSV_PATH, index=False)
print(f"[PromptLength] max_prompt_tokens={CONFIG.max_prompt_tokens}")
print(PROMPT_LENGTH_DF.round(1).to_string(index=False))
print(f"[PromptLength] Saved to {PROMPT_LENGTH_CSV_PATH}")


In [ ]:
def extract_hidden_states(model, tokenizer, prompts, layers, device, pooling, batch_size=8):
    decoder_layers = get_decoder_layers(model)
    captured = {L: [] for L in layers}
    buffers = {}

    def make_hook(layer_idx):
        def hook(module, inputs, output):
            buffers[layer_idx] = inputs[0].detach()
        return hook

    hooks = [decoder_layers[L].register_forward_hook(make_hook(L)) for L in layers]
    try:
        for i in range(0, len(prompts), batch_size):
            batch = prompts[i:i + batch_size]
            MAX_TOKENS = CONFIG.max_prompt_tokens  # Change 2: single source of truth, see Section 9b
            enc = tokenizer(
                batch,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_TOKENS,
            ).to(device)
            with torch.no_grad():
                model(**enc)
            attn = enc["attention_mask"]
            for L in layers:
                hs = buffers[L].float()
                if pooling == "last_token":
                    last_idx = attn.sum(dim=1) - 1
                    pooled = hs[torch.arange(hs.size(0)), last_idx, :]
                elif pooling == "mean":
                    mask = attn.unsqueeze(-1).float()
                    pooled = (hs * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
                elif pooling == "max":
                    mask = attn.unsqueeze(-1).bool()
                    hs_masked = hs.masked_fill(~mask, float("-inf"))
                    pooled = hs_masked.max(dim=1).values
                else:
                    raise ValueError(f"Unknown pooling strategy: {pooling}")
                captured[L].append(pooled.cpu().numpy())
            print(f"  [Extract] batch {i // batch_size + 1}/{(len(prompts) + batch_size - 1)//batch_size} done", end="\r")
    finally:
        for h in hooks:
            h.remove()

    for L in layers:
        captured[L] = np.concatenate(captured[L], axis=0)
    return captured

def benchmark_fingerprint(name, df):
    payload = json.dumps({"prompts": sorted(df["prompt"].tolist()), "pooling": CONFIG.pooling_strategy,
                           "split_ratios": CONFIG.split_ratios, "seed": CONFIG.seed}, sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]

def extract_for_benchmark_split(benchmark, split_name, df):
    cache_stage = f"hs_{benchmark}_{split_name}_extracted"
    cache_path = DIRS["activations"] / f"hs_{benchmark}_{split_name}_{CONFIG.pooling_strategy}.pkl"
    fp_key = f"{benchmark}_{split_name}"
    current_fp = benchmark_fingerprint(benchmark, df)
    stored_fp = MANIFEST["dataset_fingerprints"].get(fp_key)

    if stored_fp != current_fp and stage_done(MANIFEST, cache_stage):
        print(f"[Fingerprint][INVALIDATE] {fp_key}: data changed ({stored_fp} -> {current_fp}) — "
              f"purging its cached activations.")
        MANIFEST["completed_stages"] = [s for s in MANIFEST["completed_stages"] if s != cache_stage]
        MANIFEST["dataset_fingerprints"][fp_key] = current_fp
        save_manifest(MANIFEST)

    if stage_done(MANIFEST, cache_stage) and cache_path.exists():
        with open(cache_path, "rb") as f:
            hs = pickle.load(f)
        cached_n = next(iter(hs.values())).shape[0]
        if cached_n == len(df):
            print(f"[Resume] Loaded cached '{fp_key}' activations from {cache_path}")
            return hs
        print(f"[Extract][WARN] Cached '{fp_key}' has {cached_n} rows but current split has "
              f"{len(df)} rows — recomputing.")

    print(f"[Extract] Extracting '{fp_key}' ({len(df)} prompts)...")
    hs = extract_hidden_states(MODEL, TOKENIZER, df["prompt"].tolist(), CONFIG.layers, CONFIG.device, CONFIG.pooling_strategy)
    with open(cache_path, "wb") as f:
        pickle.dump(hs, f)
    MANIFEST["dataset_fingerprints"][fp_key] = current_fp
    mark_stage_done(MANIFEST, cache_stage)
    print(f"\n[Extract] Saved '{fp_key}' to {cache_path}")
    return hs

HS = {}  # HS[benchmark][split_name] -> {layer: ndarray}
for name in CONFIG.benchmarks_enabled:
    HS[name] = {}
    for split_name in ("train", "calibration", "test"):
        HS[name][split_name] = extract_for_benchmark_split(name, split_name, BENCH_SPLITS[name][split_name])
print("[Extract] All benchmark/split activations ready.")

## 10. Probe Training & Calibration Utilities

Mechanism unchanged from Exp017 — per-layer logistic-regression direction, ROC-calibrated
per-layer threshold, K-of-N ensemble vote. Wrapped as reusable functions here since Exp018
needs to train/calibrate/evaluate this combination many times (once per LOBO fold, once per
generalization-matrix cell) rather than once.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve, roc_auc_score, balanced_accuracy_score

def train_probe_direction(X, y, seed=42):
    clf = LogisticRegression(max_iter=2000, C=1.0, random_state=seed)
    clf.fit(X, y)
    return clf

def calibrate_layer_threshold(scores, y, target_fpr):
    fpr, tpr, thresholds = roc_curve(y, scores)
    valid = np.where(fpr <= target_fpr)[0]
    idx = valid[-1] if len(valid) else 0
    return float(thresholds[idx]), float(fpr[idx]), float(tpr[idx])

def concat_activations(hs_list):
    """hs_list: list of {layer: ndarray} dicts (e.g. one per benchmark) -> single merged dict."""
    layers = list(hs_list[0].keys())
    return {L: np.concatenate([hs[L] for hs in hs_list], axis=0) for L in layers}

def train_and_calibrate(train_df, train_hs, cal_df, cal_hs, layers, target_fpr, seed, label=""):
    """Fits one probe direction per layer on (train_df/train_hs) and calibrates its threshold
    on (cal_df/cal_hs). Returns (probes: {layer: clf}, thresholds: {layer: float}).

    Warns (does not raise) if the calibration split's benign-example count can't actually
    resolve target_fpr — e.g. target_fpr=0.02 needs >=50 benign calibration examples to be
    meaningful; with fewer, every achievable FPR value is a coarse step and the reported
    threshold is landing on the nearest one, not a real target_fpr%% threshold. Same issue
    and fix as Exp017's calibration section."""
    import math
    y_train = train_df["label_bin"].values
    y_cal = cal_df["label_bin"].values
    # ----- sanity check -----
    train_classes = np.unique(y_train)
    if len(train_classes) < 2:
        raise ValueError(
            f"{label}: training data contains only class(es) "
            f"{train_classes.tolist()}.\n"
            f"Counts:\n{train_df['label_bin'].value_counts()}"
        )
    n_safe_cal = int((y_cal == 0).sum())
    min_safe_needed = math.ceil(1 / target_fpr) if target_fpr > 0 else float("inf")
    if n_safe_cal < min_safe_needed:
        print(f"    [Calibrate][WARN] {label}: target_fpr={target_fpr} needs >={min_safe_needed} "
              f"benign calibration examples to be resolvable; only {n_safe_cal} available here. "
              f"Thresholds below are landing on the nearest resolvable step, not a real "
              f"{target_fpr*100:.0f}% FPR target — treat this fold/combo's recall as provisional.")
    probes, thresholds = {}, {}
    for L in layers:
        clf = train_probe_direction(train_hs[L], y_train, seed=seed)
        probes[L] = clf
        scores_cal = clf.decision_function(cal_hs[L])
        thresh, _, _ = calibrate_layer_threshold(scores_cal, y_cal, target_fpr) if len(set(y_cal)) > 1 else (0.0, float("nan"), float("nan"))
        thresholds[L] = thresh
    return probes, thresholds

def ensemble_predict(probes, thresholds, test_hs, layers, vote_k, layer_weights=None):
    """Returns (blocked: bool array, ensemble_score: float array [mean raw decision score
    across layers, used for AUROC], vote_total: float array)."""
    n = next(iter(test_hs.values())).shape[0]
    vote_total = np.zeros(n)
    score_sum = np.zeros(n)
    for L in layers:
        scores = probes[L].decision_function(test_hs[L])
        score_sum += scores
        flagged = scores >= thresholds[L]
        w = 1.0 if layer_weights is None else layer_weights.get(L, 1.0)
        vote_total += flagged.astype(float) * w
    blocked = vote_total >= vote_k
    ensemble_score = score_sum / len(layers)
    return blocked, ensemble_score, vote_total

def compute_metrics(y_true, y_pred_blocked, ensemble_score):
    tp = int(((y_true == 1) & (y_pred_blocked == True)).sum())
    fn = int(((y_true == 1) & (y_pred_blocked == False)).sum())
    tn = int(((y_true == 0) & (y_pred_blocked == False)).sum())
    fp = int(((y_true == 0) & (y_pred_blocked == True)).sum())
    recall = tp / (tp + fn) if (tp + fn) else float("nan")
    precision = tp / (tp + fp) if (tp + fp) else float("nan")
    if precision and recall and not np.isnan(precision) and not np.isnan(recall) and (precision + recall) > 0:
        f1 = 2 * precision * recall / (precision + recall)
    else:
        f1 = float("nan")
    fpr = fp / (fp + tn) if (fp + tn) else float("nan")
    auroc = roc_auc_score(y_true, ensemble_score) if len(set(y_true)) > 1 else float("nan")
    balanced_acc = balanced_accuracy_score(y_true, y_pred_blocked) if len(set(y_true)) > 1 else float("nan")
    return {"tp": tp, "fp": fp, "tn": tn, "fn": fn, "recall": recall, "precision": precision,
            "f1": f1, "fpr": fpr, "auroc": auroc, "balanced_accuracy": balanced_acc}

print("[Utils] Probe training / calibration / evaluation utilities ready.")

## 11. Leave-One-Benchmark-Out (LOBO) Evaluation — secondary diagnostic

**Retained for reference, no longer the primary analysis** (see the notebook overview at the
top for why: only the XSTest fold is mathematically well-posed here, since the other four
benchmarks are refusal-only and a fold that holds one of them out trains on a benign pool drawn
from the remaining benchmarks rather than testing real cross-benchmark transfer). Section 12's
transfer matrix is the primary evidence for the shared-representation hypothesis; this section's
numbers are still real held-out results and are folded into the probe registry (Section 13) and
kept in the summary report for continuity with Exp017/earlier Exp018 runs.

For each benchmark, train on the combined train+calibration rows of every OTHER benchmark,
calibrate on that combined pool's calibration rows, and test on the left-out benchmark's own
test split (never touched during training). Resumable per benchmark.

In [ ]:
LOBO_CSV_PATH = DIRS["lobo"] / "lobo_results.csv"
LOBO_PROBES_PATH = DIRS["lobo"] / "lobo_probes.pkl"

def get_benchmark_hs_and_df(name, split_name):
    return HS[name][split_name], BENCH_SPLITS[name][split_name]

def lobo_run():
    lobo_rows = []
    lobo_probes = {}
    if LOBO_CSV_PATH.exists() and stage_done(MANIFEST, "lobo_started"):
        lobo_rows = pd.read_csv(LOBO_CSV_PATH).to_dict("records")
    if LOBO_PROBES_PATH.exists():
        with open(LOBO_PROBES_PATH, "rb") as f:
            lobo_probes = pickle.load(f)
    mark_stage_done(MANIFEST, "lobo_started")

    for held_out in CONFIG.benchmarks_enabled:
        combo_key = f"lobo_holdout_{held_out}"
        if stage_done(MANIFEST, combo_key):
            print(f"[Resume] LOBO fold '{held_out}' already completed — skipping.")
            continue

        train_benches = [b for b in CONFIG.benchmarks_enabled if b != held_out]
        train_hs_list, cal_hs_list, train_df_list, cal_df_list = [], [], [], []
        for b in train_benches:
            hs_t, df_t = get_benchmark_hs_and_df(b, "train")
            hs_c, df_c = get_benchmark_hs_and_df(b, "calibration")
            train_hs_list.append(hs_t); train_df_list.append(df_t)
            cal_hs_list.append(hs_c); cal_df_list.append(df_c)

        train_hs = concat_activations(train_hs_list)
        cal_hs = concat_activations(cal_hs_list)

        train_df = pd.concat(train_df_list, ignore_index=True)
        cal_df = pd.concat(cal_df_list, ignore_index=True)

        # Skip impossible folds (training data has only one class)
        if train_df["label_bin"].nunique() < 2:
            counts = train_df["label_bin"].value_counts().to_dict()
            print(
                f"[LOBO][SKIP] held_out={held_out}: "
                f"training set contains only one class. Counts={counts}"
            )

            lobo_rows.append({
                "held_out_benchmark": held_out,
                "trained_on": "+".join(train_benches),
                "n_test": len(BENCH_SPLITS[held_out]["test"]),
                "status": "skipped_single_class"
            })

            pd.DataFrame(lobo_rows).to_csv(LOBO_CSV_PATH, index=False)
            mark_stage_done(MANIFEST, combo_key)
            continue

        probes, thresholds = train_and_calibrate(
            train_df,
            train_hs,
            cal_df,
            cal_hs,
            CONFIG.layers,
            CONFIG.target_fpr,
            CONFIG.seed,
            label=f"LOBO holdout={held_out}",
        )

        lobo_probes[held_out] = {"probes": probes, "thresholds": thresholds, "train_benchmarks": tuple(train_benches)}

        test_hs, test_df = get_benchmark_hs_and_df(held_out, "test")
        y_true = test_df["label_bin"].values
        blocked, ensemble_score, votes = ensemble_predict(probes, thresholds, test_hs, CONFIG.layers,
                                                            CONFIG.vote_k, CONFIG.layer_weights)
        metrics = compute_metrics(y_true, blocked, ensemble_score)
        row = {"held_out_benchmark": held_out, "trained_on": "+".join(train_benches),
               "n_test": len(test_df), **metrics}
        lobo_rows.append(row)
        print(f"[LOBO] held_out={held_out}: recall={metrics['recall']:.3f} precision={metrics['precision']:.3f} "
              f"f1={metrics['f1']:.3f} fpr={metrics['fpr']:.3f} auroc={metrics['auroc']:.3f} "
              f"balanced_acc={metrics['balanced_accuracy']:.3f}")

        # Save immediately after each benchmark finishes, per the spec.
        pd.DataFrame(lobo_rows).to_csv(LOBO_CSV_PATH, index=False)
        with open(LOBO_PROBES_PATH, "wb") as f:
            pickle.dump(lobo_probes, f)
        mark_stage_done(MANIFEST, combo_key)

    return pd.DataFrame(lobo_rows), lobo_probes

if CONFIG.do_lobo:
    LOBO_DF, LOBO_PROBES = lobo_run()
    print(f"\n[LOBO] Complete. {len(LOBO_DF)} folds.")
else:
    LOBO_DF, LOBO_PROBES = pd.DataFrame(), {}
    print("[LOBO] Skipped (CONFIG.do_lobo=False).")

## 12. Cross-Benchmark Transfer Matrix — primary analysis

Trains probes on the specified set of training configurations and evaluates each one against
**every** benchmark's own held-out test split, reporting the full metric set (precision, recall,
F1, FPR, AUROC, balanced accuracy) per cell — not recall alone, since recall by itself can look
strong purely by trading away precision. Train configurations, per the spec:

- `xstest` alone
- `xstest + <each other enabled benchmark>`, individually
- all enabled benchmarks together

Every benchmark's test slice was carved out in Section 8 before any training happens, so a
benchmark that also appears in a row's training combo is still evaluated on its own untouched
test rows, not on training data — every cell below is a genuine held-out number, never leakage.

**Resumability:** each (train-configuration) is its own manifest stage and its own cache
artifact (`transfer_<combo>.pkl` holding that combo's probes/thresholds, plus the running
`transfer_results.csv`, which is reloaded and appended to on restart) — a Colab disconnect
resumes from the next incomplete training configuration rather than recomputing configs already
evaluated against all five test benchmarks.

In [ ]:
TRANSFER_CSV_PATH = DIRS["transfer"] / "transfer_results.csv"
TRANSFER_PROBES_PATH = DIRS["transfer"] / "transfer_probes.pkl"
TRANSFER_HEATMAP_PATH = DIRS["figures"] / "transfer_matrix_heatmap.png"

def default_transfer_train_combos(benchmarks):
    """Change 1's specified set: [xstest], [xstest + each other benchmark individually], [all].
    Falls back to a single combo of everything enabled if 'xstest' itself got disabled/excluded."""
    benchmarks = list(benchmarks)
    if "xstest" not in benchmarks:
        return [tuple(benchmarks)]
    combos = [("xstest",)]
    for b in benchmarks:
        if b != "xstest":
            combos.append(("xstest", b))
    all_combo = tuple(benchmarks)
    if all_combo not in combos:
        combos.append(all_combo)
    return combos

TRANSFER_TRAIN_COMBOS = CONFIG.transfer_train_combos or default_transfer_train_combos(CONFIG.benchmarks_enabled)
print("[Transfer] Train configurations:", [" + ".join(c) for c in TRANSFER_TRAIN_COMBOS])

def transfer_combo_completed(combo_key):
    return stage_done(MANIFEST, f"transfer_{combo_key}")

def run_transfer_matrix():
    rows = []
    transfer_probes = {}
    if TRANSFER_CSV_PATH.exists():
        rows = pd.read_csv(TRANSFER_CSV_PATH).to_dict("records")
    if TRANSFER_PROBES_PATH.exists():
        with open(TRANSFER_PROBES_PATH, "rb") as f:
            transfer_probes = pickle.load(f)

    for combo in TRANSFER_TRAIN_COMBOS:
        combo_key = "+".join(combo)
        stage_key = f"transfer_{combo_key}"
        if transfer_combo_completed(combo_key):
            print(f"[Resume] Transfer configuration '{combo_key}' already completed — skipping.")
            continue

        train_hs_list, cal_hs_list, train_df_list, cal_df_list = [], [], [], []
        for b in combo:
            hs_t, df_t = get_benchmark_hs_and_df(b, "train")
            hs_c, df_c = get_benchmark_hs_and_df(b, "calibration")
            train_hs_list.append(hs_t); train_df_list.append(df_t)
            cal_hs_list.append(hs_c); cal_df_list.append(df_c)
        train_hs = concat_activations(train_hs_list)
        cal_hs = concat_activations(cal_hs_list)
        train_df = pd.concat(train_df_list, ignore_index=True)
        cal_df = pd.concat(cal_df_list, ignore_index=True)

        probes, thresholds = train_and_calibrate(train_df, train_hs, cal_df, cal_hs,
                                                   CONFIG.layers, CONFIG.target_fpr, CONFIG.seed,
                                                   label=f"Transfer train={combo_key}")
        transfer_probes[combo_key] = {"probes": probes, "thresholds": thresholds, "train_benchmarks": combo}

        # Drop any stale rows for this combo before re-appending (handles a partially-written
        # combo from an interrupted prior run more gracefully than pure append).
        rows = [r for r in rows if r.get("train_configuration") != combo_key]
        for test_bench in CONFIG.benchmarks_enabled:
            test_hs, test_df = get_benchmark_hs_and_df(test_bench, "test")
            y_true = test_df["label_bin"].values
            blocked, ensemble_score, votes = ensemble_predict(probes, thresholds, test_hs, CONFIG.layers,
                                                                CONFIG.vote_k, CONFIG.layer_weights)
            m = compute_metrics(y_true, blocked, ensemble_score)
            rows.append({
                "train_configuration": combo_key,
                "test_benchmark": test_bench,
                "precision": m["precision"], "recall": m["recall"], "f1": m["f1"],
                "fpr": m["fpr"], "auroc": m["auroc"], "balanced_accuracy": m["balanced_accuracy"],
            })
            print(f"[Transfer] train={combo_key:25s} test={test_bench:15s} "
                  f"recall={m['recall']:.3f} precision={m['precision']:.3f} f1={m['f1']:.3f} "
                  f"fpr={m['fpr']:.3f} auroc={m['auroc']:.3f} bal_acc={m['balanced_accuracy']:.3f}")

        # Cache immediately after each train-configuration completes, so a disconnect mid-matrix
        # resumes at the next configuration rather than restarting.
        pd.DataFrame(rows).to_csv(TRANSFER_CSV_PATH, index=False)
        with open(TRANSFER_PROBES_PATH, "wb") as f:
            pickle.dump(transfer_probes, f)
        mark_stage_done(MANIFEST, stage_key)

    return pd.DataFrame(rows), transfer_probes

TRANSFER_DF, TRANSFER_PROBES = run_transfer_matrix()
print(f"\n[Transfer] Saved to {TRANSFER_CSV_PATH}")

def transfer_pivot(metric):
    return TRANSFER_DF.pivot(index="train_configuration", columns="test_benchmark", values=metric) \
        .reindex(["+".join(c) for c in TRANSFER_TRAIN_COMBOS])

TRANSFER_DF.round(3)


In [ ]:
def plot_transfer_heatmaps():
    metrics = ["recall", "auroc", "balanced_accuracy"]
    fig, axes = plt.subplots(1, len(metrics), figsize=(6 * len(metrics), 5))
    for ax, metric in zip(axes, metrics):
        pv = transfer_pivot(metric)
        im = ax.imshow(pv.values, vmin=0, vmax=1, cmap="viridis")
        ax.set_xticks(range(len(pv.columns))); ax.set_xticklabels(pv.columns, rotation=45, ha="right", fontsize=8)
        ax.set_yticks(range(len(pv.index))); ax.set_yticklabels(pv.index, fontsize=8)
        for i in range(pv.shape[0]):
            for j in range(pv.shape[1]):
                val = pv.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f"{val:.2f}", ha="center", va="center",
                            color="white" if val < 0.6 else "black", fontsize=7)
        ax.set_title(metric)
        ax.set_xlabel("test benchmark"); ax.set_ylabel("train configuration")
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle("Cross-Benchmark Transfer Matrix")
    fig.tight_layout()
    fig.savefig(TRANSFER_HEATMAP_PATH, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"[Transfer] Saved heatmap to {TRANSFER_HEATMAP_PATH}")

plot_transfer_heatmaps()


## 13. Probe Similarity Analysis

Registers every probe trained in Sections 11-12 (LOBO folds + transfer-matrix train
configurations), one entry per (training-configuration, layer), and computes pairwise cosine
similarity **within each layer** (comparing a layer-19 probe to a layer-19 probe, not across
layers — a cross-layer comparison isn't meaningful since different layers' representations live
in different bases). Answers: do probes trained on different benchmark combinations converge
toward the same direction? Section 14c puts these similarity numbers against a random-direction
baseline to establish whether any convergence found here is bigger than high-dimensional
geometry would produce by chance alone.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

PROBE_SIM_CSV_PATH = DIRS["similarity"] / "probe_similarity.csv"
PROBE_SIM_HEATMAP_PATH = DIRS["figures"] / "probe_similarity_heatmap.png"

def build_probe_registry():
    registry = []  # each: {combo_name, benchmarks, layer, weight, bias}
    for held_out, entry in LOBO_PROBES.items():
        combo_name = f"LOBO_holdout_{held_out}"
        for L, clf in entry["probes"].items():
            registry.append({"combo_name": combo_name, "benchmarks": entry["train_benchmarks"],
                              "layer": L, "weight": clf.coef_[0].copy(), "bias": float(clf.intercept_[0])})
    for combo_name, entry in TRANSFER_PROBES.items():
        for L, clf in entry["probes"].items():
            registry.append({"combo_name": f"Transfer_{combo_name}", "benchmarks": entry["train_benchmarks"],
                              "layer": L, "weight": clf.coef_[0].copy(), "bias": float(clf.intercept_[0])})
    return registry

PROBE_REGISTRY = build_probe_registry()
with open(DIRS["probes"] / "probe_registry.pkl", "wb") as f:
    pickle.dump(PROBE_REGISTRY, f)
print(f"[ProbeSimilarity] Registered {len(PROBE_REGISTRY)} probes across LOBO folds + transfer matrix.")

sim_rows = []
for L in CONFIG.layers:
    layer_entries = [e for e in PROBE_REGISTRY if e["layer"] == L]
    if len(layer_entries) < 2:
        continue
    W = np.stack([e["weight"] for e in layer_entries])
    sim_matrix = cosine_similarity(W)
    for i, ei in enumerate(layer_entries):
        for j, ej in enumerate(layer_entries):
            if j <= i:
                continue
            sim_rows.append({"layer": L, "combo_a": ei["combo_name"], "combo_b": ej["combo_name"],
                              "cosine_similarity": float(sim_matrix[i, j])})

PROBE_SIM_DF = pd.DataFrame(sim_rows)
PROBE_SIM_DF.to_csv(PROBE_SIM_CSV_PATH, index=False)
print(f"[ProbeSimilarity] Saved {len(PROBE_SIM_DF)} pairwise comparisons to {PROBE_SIM_CSV_PATH}")
if len(PROBE_SIM_DF):
    print(f"[ProbeSimilarity] Overall mean cosine similarity: {PROBE_SIM_DF['cosine_similarity'].mean():.4f} "
          f"(1.0 = identical direction, 0 = orthogonal/unrelated, <0 = opposing)")

import matplotlib.pyplot as plt

def plot_probe_similarity_heatmaps():
    n_layers = len(CONFIG.layers)
    fig, axes = plt.subplots(1, n_layers, figsize=(5 * n_layers, 4.5))
    if n_layers == 1:
        axes = [axes]
    for ax, L in zip(axes, CONFIG.layers):
        layer_entries = [e for e in PROBE_REGISTRY if e["layer"] == L]
        if len(layer_entries) < 2:
            ax.set_title(f"layer {L} (insufficient probes)")
            continue
        W = np.stack([e["weight"] for e in layer_entries])
        sim_matrix = cosine_similarity(W)
        labels = [e["combo_name"].replace("LOBO_holdout_", "LOBO~").replace("Transfer_", "") for e in layer_entries]
        im = ax.imshow(sim_matrix, vmin=-1, vmax=1, cmap="RdBu_r")
        ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=90, fontsize=6)
        ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=6)
        ax.set_title(f"layer {L}")
        fig.colorbar(im, ax=ax, fraction=0.046)
    fig.suptitle("Probe direction cosine similarity across training combinations, per layer")
    fig.tight_layout()
    fig.savefig(PROBE_SIM_HEATMAP_PATH, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"[ProbeSimilarity] Saved heatmap to {PROBE_SIM_HEATMAP_PATH}")

plot_probe_similarity_heatmaps()

## 13b. Probe Geometry Analysis (PCA / UMAP / Hierarchical Clustering of probe directions)

Section 16 later runs PCA/UMAP over individual **activation vectors** (one point per prompt) to
visualize the safe/refusal split. This section is different: each point here is an entire
**probe direction** (one point per training configuration, at a fixed layer) — i.e. it visualizes
how the *learned classifiers themselves* relate to each other in weight-space, complementing the
pairwise cosine-similarity numbers in Section 13 with a 2D layout and a dendrogram that group
similar directions together, rather than reading a full N×N similarity matrix.

Runs at a single representative layer (`CONFIG.probe_geometry_layer`, default = middle of
`CONFIG.layers`) since directions from different layers live in different bases and shouldn't be
projected together.

In [ ]:
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import pdist

PROBE_PCA_CSV_PATH = DIRS["geometry"] / "probe_pca.csv"
PROBE_UMAP_CSV_PATH = DIRS["geometry"] / "probe_umap.csv"
PROBE_DENDRO_CSV_PATH = DIRS["geometry"] / "probe_dendrogram_linkage.csv"
PROBE_PCA_FIG_PATH = DIRS["figures"] / "probe_pca.png"
PROBE_UMAP_FIG_PATH = DIRS["figures"] / "probe_umap.png"
PROBE_DENDRO_FIG_PATH = DIRS["figures"] / "probe_dendrogram.png"

GEOMETRY_LAYER = CONFIG.probe_geometry_layer or CONFIG.layers[len(CONFIG.layers) // 2]
GEOMETRY_STAGE_KEY = f"probe_geometry_layer{GEOMETRY_LAYER}_n{len(PROBE_REGISTRY)}"

def run_probe_geometry():
    if stage_done(MANIFEST, GEOMETRY_STAGE_KEY) and PROBE_PCA_CSV_PATH.exists():
        print(f"[Resume] Probe geometry (layer {GEOMETRY_LAYER}) already completed — loading cached coordinates.")
        pca_df = pd.read_csv(PROBE_PCA_CSV_PATH)
        umap_df = pd.read_csv(PROBE_UMAP_CSV_PATH) if PROBE_UMAP_CSV_PATH.exists() else pd.DataFrame()
        return pca_df, umap_df

    entries = [e for e in PROBE_REGISTRY if e["layer"] == GEOMETRY_LAYER]
    if len(entries) < 2:
        print(f"[ProbeGeometry][WARN] fewer than 2 probes at layer {GEOMETRY_LAYER} — skipping.")
        return pd.DataFrame(), pd.DataFrame()

    W = np.stack([e["weight"] for e in entries])
    labels = [e["combo_name"] for e in entries]

    # --- PCA ---
    n_components = min(2, W.shape[0] - 1, W.shape[1])
    pca = PCA(n_components=n_components, random_state=CONFIG.seed)
    coords = pca.fit_transform(W)
    pc2 = coords[:, 1] if coords.shape[1] > 1 else np.zeros(coords.shape[0])
    pca_df = pd.DataFrame({"combo_name": labels, "pc1": coords[:, 0], "pc2": pc2})
    pca_df.to_csv(PROBE_PCA_CSV_PATH, index=False)

    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(pca_df["pc1"], pca_df["pc2"], s=40)
    for _, r in pca_df.iterrows():
        ax.annotate(r["combo_name"], (r["pc1"], r["pc2"]), fontsize=6, xytext=(3, 3), textcoords="offset points")
    ax.set_title(f"PCA of probe directions (layer {GEOMETRY_LAYER})")
    ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
    fig.tight_layout()
    fig.savefig(PROBE_PCA_FIG_PATH, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[ProbeGeometry] PCA explained variance ratio: {pca.explained_variance_ratio_}")
    print(f"[ProbeGeometry] Saved {PROBE_PCA_CSV_PATH} and {PROBE_PCA_FIG_PATH}")

    # --- UMAP ---
    umap_df = pd.DataFrame()
    if UMAP_AVAILABLE and W.shape[0] >= 4:
        import umap
        reducer = umap.UMAP(n_components=2, random_state=CONFIG.seed, n_neighbors=min(5, W.shape[0] - 1))
        ucoords = reducer.fit_transform(W)
        umap_df = pd.DataFrame({"combo_name": labels, "umap1": ucoords[:, 0], "umap2": ucoords[:, 1]})
        umap_df.to_csv(PROBE_UMAP_CSV_PATH, index=False)

        fig, ax = plt.subplots(figsize=(7, 6))
        ax.scatter(umap_df["umap1"], umap_df["umap2"], s=40, color="tab:orange")
        for _, r in umap_df.iterrows():
            ax.annotate(r["combo_name"], (r["umap1"], r["umap2"]), fontsize=6, xytext=(3, 3), textcoords="offset points")
        ax.set_title(f"UMAP of probe directions (layer {GEOMETRY_LAYER})")
        fig.tight_layout()
        fig.savefig(PROBE_UMAP_FIG_PATH, dpi=150, bbox_inches="tight")
        plt.close(fig)
        print(f"[ProbeGeometry] Saved {PROBE_UMAP_CSV_PATH} and {PROBE_UMAP_FIG_PATH}")
    else:
        print("[ProbeGeometry] Skipping UMAP (umap-learn unavailable, or fewer than 4 probes at this layer).")

    # --- Hierarchical clustering / dendrogram (cosine distance) ---
    dist = pdist(W, metric="cosine")
    Z = linkage(dist, method="average")
    fig, ax = plt.subplots(figsize=(max(8, 0.4 * len(labels)), 5))
    dendrogram(Z, labels=labels, leaf_rotation=90, leaf_font_size=7, ax=ax)
    ax.set_title(f"Hierarchical clustering of probe directions (layer {GEOMETRY_LAYER}, cosine distance)")
    ax.set_ylabel("cosine distance")
    fig.tight_layout()
    fig.savefig(PROBE_DENDRO_FIG_PATH, dpi=150, bbox_inches="tight")
    plt.close(fig)
    pd.DataFrame(Z, columns=["cluster_a", "cluster_b", "distance", "n_items"]).to_csv(PROBE_DENDRO_CSV_PATH, index=False)
    print(f"[ProbeGeometry] Saved {PROBE_DENDRO_CSV_PATH} and {PROBE_DENDRO_FIG_PATH}")

    mark_stage_done(MANIFEST, GEOMETRY_STAGE_KEY)
    return pca_df, umap_df

PROBE_PCA_DF, PROBE_UMAP_DF = run_probe_geometry()


## 14. Layer Stability Analysis

Trains a probe at each of layers 19-22 on the SAME reference training pool (all enabled
benchmarks combined) and measures pairwise cosine similarity between layers' directions. High
similarity across adjacent layers would suggest the "unsafe intent" direction is a fairly
stable rotation of itself through the residual stream rather than a qualitatively different
representation at each layer.

In [ ]:
LAYER_SIM_CSV_PATH = DIRS["similarity"] / "layer_similarity.csv"

all_train_hs = concat_activations([HS[b]["train"] for b in CONFIG.benchmarks_enabled])
all_train_df = pd.concat([BENCH_SPLITS[b]["train"] for b in CONFIG.benchmarks_enabled], ignore_index=True)
y_all_train = all_train_df["label_bin"].values

REFERENCE_LAYER_PROBES = {}
for L in CONFIG.layers:
    REFERENCE_LAYER_PROBES[L] = train_probe_direction(all_train_hs[L], y_all_train, seed=CONFIG.seed)

layer_sim_rows = []
layers_list = list(CONFIG.layers)
W_by_layer = {L: REFERENCE_LAYER_PROBES[L].coef_[0] for L in layers_list}
for i, La in enumerate(layers_list):
    for j, Lb in enumerate(layers_list):
        if j <= i:
            continue
        sim = float(cosine_similarity(W_by_layer[La].reshape(1, -1), W_by_layer[Lb].reshape(1, -1))[0, 0])
        layer_sim_rows.append({"layer_a": La, "layer_b": Lb, "cosine_similarity": sim})

LAYER_SIM_DF = pd.DataFrame(layer_sim_rows)
LAYER_SIM_DF.to_csv(LAYER_SIM_CSV_PATH, index=False)
print(f"[LayerStability] Saved to {LAYER_SIM_CSV_PATH}")
print(LAYER_SIM_DF.round(4))
LAYER_SIM_HEATMAP_PATH = DIRS["figures"] / "layer_similarity_heatmap.png"

def plot_layer_similarity_heatmap():
    n = len(layers_list)
    mat = np.eye(n)
    for _, r in LAYER_SIM_DF.iterrows():
        i, j = layers_list.index(int(r["layer_a"])), layers_list.index(int(r["layer_b"]))
        mat[i, j] = mat[j, i] = r["cosine_similarity"]
    fig, ax = plt.subplots(figsize=(5, 4.5))
    im = ax.imshow(mat, vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(n)); ax.set_xticklabels(layers_list)
    ax.set_yticks(range(n)); ax.set_yticklabels(layers_list)
    for i in range(n):
        for j in range(n):
            ax.text(j, i, f"{mat[i, j]:.2f}", ha="center", va="center",
                    color="white" if abs(mat[i, j]) > 0.6 else "black", fontsize=8)
    ax.set_title("Layer stability: reference-probe cosine similarity across layers")
    ax.set_xlabel("layer"); ax.set_ylabel("layer")
    fig.colorbar(im, ax=ax, fraction=0.046)
    fig.tight_layout()
    fig.savefig(LAYER_SIM_HEATMAP_PATH, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[LayerStability] Saved heatmap to {LAYER_SIM_HEATMAP_PATH}")

plot_layer_similarity_heatmap()


## 14b. Permutation Control (negative control)

Negative control: independently shuffle the reference training pool's labels (same pool used in
Section 14 — all enabled benchmarks' train rows combined), retrain a probe per layer on the
shuffled labels, and evaluate it on the SAME real, unshuffled combined test labels used
throughout this notebook. Repeated `CONFIG.n_permutations` times per layer.

**What a pass/fail here looks like:** if shuffled-label AUROC clusters near 0.5 (chance) while
the real-label AUROC sits well above it, the real probe's signal is tied to the actual
safe/refusal distinction rather than some other exploitable structure (e.g. benchmark-of-origin
correlating with label by coincidence in this particular sample). If shuffled AUROC is *not*
near chance, something in the data or pipeline (e.g. subtle train/test leakage) is inflating
every probe's apparent performance, including the ones already reported above.

In [ ]:
PERMUTATION_CSV_PATH = DIRS["controls"] / "permutation_results.csv"
PERMUTATION_FIG_PATH = DIRS["figures"] / "permutation_histogram.png"

all_test_hs_ref = concat_activations([HS[b]["test"] for b in CONFIG.benchmarks_enabled])
all_test_df_ref = pd.concat([BENCH_SPLITS[b]["test"] for b in CONFIG.benchmarks_enabled], ignore_index=True)
y_all_test_ref = all_test_df_ref["label_bin"].values

PERMUTATION_STAGE_KEY = f"permutation_control_n{CONFIG.n_permutations}"

def run_permutation_control():
    if stage_done(MANIFEST, PERMUTATION_STAGE_KEY) and PERMUTATION_CSV_PATH.exists():
        print(f"[Resume] Permutation control (n={CONFIG.n_permutations}) already completed.")
        return pd.read_csv(PERMUTATION_CSV_PATH)

    rows = []
    # Real (unshuffled) AUROC per layer, using the reference probes already trained in Section 14.
    for L in CONFIG.layers:
        scores = REFERENCE_LAYER_PROBES[L].decision_function(all_test_hs_ref[L])
        auroc = roc_auc_score(y_all_test_ref, scores) if len(set(y_all_test_ref)) > 1 else float("nan")
        rows.append({"run_type": "real", "permutation_index": -1, "layer": L, "auroc": auroc})

    # Shuffled-label AUROC, N repeats per layer.
    for p_idx in range(CONFIG.n_permutations):
        rng_seed = CONFIG.seed * 1000 + p_idx
        shuffled_y = np.random.default_rng(rng_seed).permutation(y_all_train)
        for L in CONFIG.layers:
            if len(set(shuffled_y)) < 2:
                continue
            clf = train_probe_direction(all_train_hs[L], shuffled_y, seed=rng_seed)
            scores = clf.decision_function(all_test_hs_ref[L])
            auroc = roc_auc_score(y_all_test_ref, scores) if len(set(y_all_test_ref)) > 1 else float("nan")
            rows.append({"run_type": "shuffled", "permutation_index": p_idx, "layer": L, "auroc": auroc})
        print(f"  [Permutation] {p_idx + 1}/{CONFIG.n_permutations} complete", end="\r")
    print()

    df = pd.DataFrame(rows)
    df.to_csv(PERMUTATION_CSV_PATH, index=False)
    mark_stage_done(MANIFEST, PERMUTATION_STAGE_KEY)
    print(f"[Permutation] Saved to {PERMUTATION_CSV_PATH}")
    return df

PERMUTATION_DF = run_permutation_control()

real_auroc = PERMUTATION_DF[PERMUTATION_DF["run_type"] == "real"].set_index("layer")["auroc"]
shuffled_auroc = PERMUTATION_DF[PERMUTATION_DF["run_type"] == "shuffled"]
print("[Permutation] Real AUROC by layer:\n", real_auroc.round(4))
if len(shuffled_auroc):
    print("[Permutation] Shuffled AUROC — mean/std by layer:\n",
          shuffled_auroc.groupby("layer")["auroc"].agg(["mean", "std"]).round(4))

def plot_permutation_histogram():
    layers_ = list(CONFIG.layers)
    fig, axes = plt.subplots(1, len(layers_), figsize=(5 * len(layers_), 4), sharey=True)
    if len(layers_) == 1:
        axes = [axes]
    for ax, L in zip(axes, layers_):
        vals = shuffled_auroc[shuffled_auroc["layer"] == L]["auroc"].dropna()
        ax.hist(vals, bins=15, color="tab:gray", alpha=0.8, label="shuffled labels")
        if L in real_auroc.index and not np.isnan(real_auroc[L]):
            ax.axvline(real_auroc[L], color="tab:red", linewidth=2, label="real labels")
        ax.axvline(0.5, color="black", linestyle="--", linewidth=1, label="chance (0.5)")
        ax.set_title(f"layer {L}")
        ax.set_xlabel("AUROC")
        ax.legend(fontsize=7)
    axes[0].set_ylabel("count")
    fig.suptitle(f"Permutation control: real vs. shuffled-label AUROC (n={CONFIG.n_permutations} shuffles/layer)")
    fig.tight_layout()
    fig.savefig(PERMUTATION_FIG_PATH, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"[Permutation] Saved histogram to {PERMUTATION_FIG_PATH}")

plot_permutation_histogram()


## 14c. Random Direction Baseline (negative control)

Second negative control: draw random unit vectors matching the probes' dimensionality (the
residual-stream hidden size) and compute their pairwise cosine similarities, exactly as Section
13 does for real probes. In a high-dimensional space, independently drawn random vectors already
have some non-zero expected cosine similarity purely from concentration-of-measure effects — so
"probe cosine similarities are positive" isn't by itself evidence of a shared representation.
This section makes that comparison explicit: real probe-pair similarity has to clear the random
baseline, not just clear zero.

In [ ]:
RANDOM_BASELINE_CSV_PATH = DIRS["controls"] / "random_direction_baseline.csv"
RANDOM_BASELINE_FIG_PATH = DIRS["figures"] / "random_baseline_comparison.png"
RANDOM_BASELINE_STAGE_KEY = "random_direction_baseline"

def run_random_direction_baseline():
    if stage_done(MANIFEST, RANDOM_BASELINE_STAGE_KEY) and RANDOM_BASELINE_CSV_PATH.exists():
        print("[Resume] Random-direction baseline already completed.")
        return pd.read_csv(RANDOM_BASELINE_CSV_PATH)

    rng = np.random.default_rng(CONFIG.seed)
    dim = next(iter(all_train_hs.values())).shape[1]
    n_random = max(len(PROBE_REGISTRY), 20)
    random_vectors = rng.normal(size=(n_random, dim))
    random_vectors /= np.linalg.norm(random_vectors, axis=1, keepdims=True)
    sim_matrix = cosine_similarity(random_vectors)

    rows = []
    for i in range(n_random):
        for j in range(n_random):
            if j <= i:
                continue
            rows.append({"vector_a": i, "vector_b": j, "cosine_similarity": float(sim_matrix[i, j])})
    df = pd.DataFrame(rows)
    df.to_csv(RANDOM_BASELINE_CSV_PATH, index=False)
    mark_stage_done(MANIFEST, RANDOM_BASELINE_STAGE_KEY)
    print(f"[RandomBaseline] {n_random} random {dim}-dim unit vectors -> {len(df)} pairwise "
          f"comparisons. Saved to {RANDOM_BASELINE_CSV_PATH}")
    return df

RANDOM_BASELINE_DF = run_random_direction_baseline()

print(f"[RandomBaseline] Random-pair mean cosine similarity: "
      f"{RANDOM_BASELINE_DF['cosine_similarity'].mean():.4f} (std={RANDOM_BASELINE_DF['cosine_similarity'].std():.4f})")
if len(PROBE_SIM_DF):
    print(f"[RandomBaseline] Real probe-pair mean cosine similarity: "
          f"{PROBE_SIM_DF['cosine_similarity'].mean():.4f} (std={PROBE_SIM_DF['cosine_similarity'].std():.4f})")

fig, ax = plt.subplots(figsize=(7, 5))
ax.hist(RANDOM_BASELINE_DF["cosine_similarity"], bins=30, alpha=0.6, density=True,
        label="random vector pairs", color="tab:gray")
if len(PROBE_SIM_DF):
    ax.hist(PROBE_SIM_DF["cosine_similarity"], bins=30, alpha=0.6, density=True,
            label="real probe pairs", color="tab:blue")
ax.axvline(0, color="black", linestyle="--", linewidth=1)
ax.set_xlabel("cosine similarity"); ax.set_ylabel("density")
ax.set_title("Probe-direction similarity vs. random-direction baseline")
ax.legend()
fig.tight_layout()
fig.savefig(RANDOM_BASELINE_FIG_PATH, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"[RandomBaseline] Saved comparison figure to {RANDOM_BASELINE_FIG_PATH}")


## 15. Seed Stability Analysis

**Design note:** sklearn's default `lbfgs` solver is a deterministic convex optimizer — fitting
the exact same data at different `random_state` values yields an identical probe, which would
make a naive seed-stability test trivially report cosine similarity = 1.0 for every pair and
tell you nothing. The real question — does the learned direction depend on which examples you
happened to draw, or does it converge regardless — is measured here by **bootstrap-resampling**
the reference training pool at each seed (same size, sampled with replacement) and comparing
the resulting probes. This is what "seed stability" should mean for a deterministic solver.

In [ ]:
SEED_STABILITY_CSV_PATH = DIRS["similarity"] / "seed_stability.csv"

def bootstrap_probe(X, y, seed, n_samples=None):
    rng = np.random.default_rng(seed)
    n = len(y) if n_samples is None else n_samples
    idx = rng.integers(0, len(y), n)
    return train_probe_direction(X[idx], y[idx], seed=seed)

seed_probes = {L: {} for L in CONFIG.layers}
for L in CONFIG.layers:
    for s in CONFIG.stability_seeds:
        seed_probes[L][s] = bootstrap_probe(all_train_hs[L], y_all_train, seed=s)

seed_stability_rows = []
for L in CONFIG.layers:
    seeds_list = list(CONFIG.stability_seeds)
    for i, sa in enumerate(seeds_list):
        for j, sb in enumerate(seeds_list):
            if j <= i:
                continue
            wa, wb = seed_probes[L][sa].coef_[0], seed_probes[L][sb].coef_[0]
            sim = float(cosine_similarity(wa.reshape(1, -1), wb.reshape(1, -1))[0, 0])
            seed_stability_rows.append({"layer": L, "seed_a": sa, "seed_b": sb, "cosine_similarity": sim})

SEED_STABILITY_DF = pd.DataFrame(seed_stability_rows)
SEED_STABILITY_DF.to_csv(SEED_STABILITY_CSV_PATH, index=False)
print(f"[SeedStability] Saved to {SEED_STABILITY_CSV_PATH}")
print(SEED_STABILITY_DF.round(4))
if len(SEED_STABILITY_DF):
    print(f"[SeedStability] Mean cosine similarity across bootstrap seeds: "
          f"{SEED_STABILITY_DF['cosine_similarity'].mean():.4f} (closer to 1.0 = more stable direction)")
SEED_STABILITY_SUMMARY_CSV_PATH = DIRS["similarity"] / "seed_stability_summary.csv"

def summarize_seed_stability(df, n_bootstrap_ci=2000, seed=CONFIG.seed):
    """Per layer: mean cosine similarity, std, and a bootstrap 95% CI over the pairwise
    similarity values (Change 7). With only 3 stability seeds there are just 3 pairs per layer,
    so the CI is necessarily wide — it's reported honestly rather than masked."""
    rng = np.random.default_rng(seed)
    rows = []
    for L, g in df.groupby("layer"):
        vals = g["cosine_similarity"].values
        boot_means = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_bootstrap_ci)]
        lo, hi = np.percentile(boot_means, [2.5, 97.5])
        rows.append({
            "layer": L, "n_pairs": len(vals),
            "mean_cosine": float(vals.mean()), "std_cosine": float(vals.std()),
            "ci95_lower": float(lo), "ci95_upper": float(hi),
        })
    return pd.DataFrame(rows)

SEED_STABILITY_SUMMARY_DF = summarize_seed_stability(SEED_STABILITY_DF, n_bootstrap_ci=CONFIG.n_bootstrap)
SEED_STABILITY_SUMMARY_DF.to_csv(SEED_STABILITY_SUMMARY_CSV_PATH, index=False)
print(f"[SeedStability] Per-layer summary (mean / std / 95% CI):\n{SEED_STABILITY_SUMMARY_DF.round(4).to_string(index=False)}")
print(f"[SeedStability] Saved to {SEED_STABILITY_SUMMARY_CSV_PATH}")


## 16. PCA / UMAP Representation Visualization\n\nHeld-out (test-split) activations at the middle configured layer, projected to 2D and colored by both label and benchmark of origin.

In [ ]:
from sklearn.decomposition import PCA

PCA_FIG_PATH = DIRS["figures"] / "activation_pca.png"
UMAP_FIG_PATH = DIRS["figures"] / "activation_umap.png"
VIZ_LAYER = CONFIG.layers[len(CONFIG.layers) // 2]

all_test_hs_viz = concat_activations([HS[b]["test"] for b in CONFIG.benchmarks_enabled])[VIZ_LAYER]
all_test_df_viz = pd.concat([BENCH_SPLITS[b]["test"] for b in CONFIG.benchmarks_enabled], ignore_index=True)

def plot_2d_projection(coords, df, title, out_path):
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    label_colors = {"safe": "tab:blue", "refusal": "tab:red"}
    for lbl, color in label_colors.items():
        mask = (df["label"] == lbl).values
        axes[0].scatter(coords[mask, 0], coords[mask, 1], s=14, alpha=0.6, label=lbl, color=color)
    axes[0].set_title(f"{title} — colored by label"); axes[0].legend()

    benchmarks_present = df["benchmark"].unique()
    cmap = plt.get_cmap("tab10")
    for i, b in enumerate(benchmarks_present):
        mask = (df["benchmark"] == b).values
        axes[1].scatter(coords[mask, 0], coords[mask, 1], s=14, alpha=0.6, label=b, color=cmap(i % 10))
    axes[1].set_title(f"{title} — colored by benchmark"); axes[1].legend(fontsize=7)

    fig.tight_layout()
    fig.savefig(out_path, bbox_inches="tight", dpi=150)
    plt.close(fig)
    print(f"  [Figure] Saved {out_path}")

pca = PCA(n_components=2, random_state=CONFIG.seed)
pca_coords = pca.fit_transform(all_test_hs_viz)
plot_2d_projection(pca_coords, all_test_df_viz, f"PCA (layer {VIZ_LAYER}, held-out test)", PCA_FIG_PATH)
print(f"[PCA] Explained variance ratio (2 components): {pca.explained_variance_ratio_}")

if UMAP_AVAILABLE:
    import umap
    reducer = umap.UMAP(n_components=2, random_state=CONFIG.seed)
    umap_coords = reducer.fit_transform(all_test_hs_viz)
    plot_2d_projection(umap_coords, all_test_df_viz, f"UMAP (layer {VIZ_LAYER}, held-out test)", UMAP_FIG_PATH)
else:
    print("[UMAP] umap-learn not available — skipping activation_umap.png (PCA above still produced).")

## 17. Save Outputs

In [ ]:
import shutil

REQUIRED_OUTPUTS = {
    "exp018_config.json": DIRS["root"] / "exp018_config.json",
    "prompt_length_report.csv": PROMPT_LENGTH_CSV_PATH,
    "lobo_results.csv": LOBO_CSV_PATH,
    "transfer_results.csv": TRANSFER_CSV_PATH,
    "transfer_matrix_heatmap.png": TRANSFER_HEATMAP_PATH,
    "probe_similarity.csv": PROBE_SIM_CSV_PATH,
    "probe_similarity_heatmap.png": PROBE_SIM_HEATMAP_PATH,
    "probe_pca.csv": PROBE_PCA_CSV_PATH,
    "probe_pca.png": PROBE_PCA_FIG_PATH,
    "probe_dendrogram_linkage.csv": PROBE_DENDRO_CSV_PATH,
    "probe_dendrogram.png": PROBE_DENDRO_FIG_PATH,
    "layer_similarity.csv": LAYER_SIM_CSV_PATH,
    "layer_similarity_heatmap.png": LAYER_SIM_HEATMAP_PATH,
    "permutation_results.csv": PERMUTATION_CSV_PATH,
    "permutation_histogram.png": PERMUTATION_FIG_PATH,
    "random_direction_baseline.csv": RANDOM_BASELINE_CSV_PATH,
    "random_baseline_comparison.png": RANDOM_BASELINE_FIG_PATH,
    "seed_stability.csv": SEED_STABILITY_CSV_PATH,
    "seed_stability_summary.csv": SEED_STABILITY_SUMMARY_CSV_PATH,
    "activation_pca.png": PCA_FIG_PATH,
}
# UMAP figures are conditional on umap-learn being installed — only require them if produced.
if PROBE_UMAP_CSV_PATH.exists():
    REQUIRED_OUTPUTS["probe_umap.csv"] = PROBE_UMAP_CSV_PATH
    REQUIRED_OUTPUTS["probe_umap.png"] = PROBE_UMAP_FIG_PATH
if UMAP_FIG_PATH.exists():
    REQUIRED_OUTPUTS["activation_umap.png"] = UMAP_FIG_PATH

missing = [name for name, p in REQUIRED_OUTPUTS.items() if not p.exists()]
if missing:
    print(f"[Save][WARN] Missing expected outputs: {missing}")
else:
    print("[Save] All required output files are present.")

zip_path = DIRS["root"] / f"exp018_run_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
shutil.make_archive(str(zip_path), "zip", root_dir=DIRS["root"])
print(f"[Save] Archived full run to {zip_path}.zip")

if IN_COLAB:
    try:
        from google.colab import files
        if USE_LOCAL_STORAGE:
            print(f"[Save] Drive unavailable this session — triggering browser download of {zip_path}.zip")
            files.download(f"{zip_path}.zip")
    except Exception as e:
        print(f"[Save][WARN] Could not trigger automatic download ({e}). Manually download "
              f"{zip_path}.zip from the Colab file browser if Drive is unavailable.")


## 18. Final Results Report

In [ ]:
def build_summary():
    best_row = None
    largest_transfer_gap = None
    transfer_matrix_summary = {}
    if len(TRANSFER_DF):
        pv_recall = transfer_pivot("recall")
        pv_auroc = transfer_pivot("auroc")
        mean_recall_per_combo = pv_recall.mean(axis=1)
        best_combo = mean_recall_per_combo.idxmax()
        best_row = {"train_configuration": best_combo, "mean_recall_across_all_test_benchmarks": float(mean_recall_per_combo.max())}
        gaps = pv_recall.max(axis=1) - pv_recall.min(axis=1)
        largest_transfer_gap = {"train_configuration": str(gaps.idxmax()), "gap": float(gaps.max())}
        transfer_matrix_summary = {
            "train_configurations": list(pv_recall.index),
            "test_benchmarks": list(pv_recall.columns),
            "mean_recall_overall": float(pv_recall.values[~np.isnan(pv_recall.values)].mean()) if pv_recall.size else None,
            "mean_auroc_overall": float(pv_auroc.values[~np.isnan(pv_auroc.values)].mean()) if pv_auroc.size else None,
        }

    lobo_summary = {}
    if len(LOBO_DF) and "recall" in LOBO_DF.columns:
        valid_lobo = LOBO_DF.dropna(subset=["recall"])
        if len(valid_lobo):
            lobo_summary = {
                "best_unseen_benchmark_recall": {
                    "benchmark": str(valid_lobo.loc[valid_lobo["recall"].idxmax(), "held_out_benchmark"]),
                    "recall": float(valid_lobo["recall"].max()),
                },
                "worst_unseen_benchmark_recall": {
                    "benchmark": str(valid_lobo.loc[valid_lobo["recall"].idxmin(), "held_out_benchmark"]),
                    "recall": float(valid_lobo["recall"].min()),
                },
                "average_recall": float(valid_lobo["recall"].mean()),
                "average_fpr": float(valid_lobo["fpr"].mean()) if "fpr" in valid_lobo else None,
                "average_auroc": float(valid_lobo["auroc"].mean()) if "auroc" in valid_lobo else None,
            }

    permutation_summary = {}
    if len(PERMUTATION_DF):
        real_ = PERMUTATION_DF[PERMUTATION_DF["run_type"] == "real"]
        shuf_ = PERMUTATION_DF[PERMUTATION_DF["run_type"] == "shuffled"]
        permutation_summary = {
            "n_permutations": CONFIG.n_permutations,
            "real_auroc_by_layer": real_.set_index("layer")["auroc"].round(4).to_dict(),
            "shuffled_auroc_mean_by_layer": shuf_.groupby("layer")["auroc"].mean().round(4).to_dict() if len(shuf_) else {},
            "shuffled_auroc_std_by_layer": shuf_.groupby("layer")["auroc"].std().round(4).to_dict() if len(shuf_) else {},
        }

    random_baseline_summary = {}
    if len(RANDOM_BASELINE_DF):
        random_baseline_summary = {
            "random_pair_mean_cosine": float(RANDOM_BASELINE_DF["cosine_similarity"].mean()),
            "random_pair_std_cosine": float(RANDOM_BASELINE_DF["cosine_similarity"].std()),
            "real_probe_pair_mean_cosine": float(PROBE_SIM_DF["cosine_similarity"].mean()) if len(PROBE_SIM_DF) else None,
            "real_exceeds_random_baseline": (
                bool(PROBE_SIM_DF["cosine_similarity"].mean() > RANDOM_BASELINE_DF["cosine_similarity"].mean())
                if len(PROBE_SIM_DF) else None
            ),
        }

    probe_geometry_summary = {}
    if len(PROBE_PCA_DF):
        probe_geometry_summary = {
            "geometry_layer": GEOMETRY_LAYER,
            "n_probes_at_layer": len(PROBE_PCA_DF),
            "pca_coordinates_file": "probe_pca.csv",
            "umap_coordinates_file": "probe_umap.csv" if len(PROBE_UMAP_DF) else None,
            "dendrogram_linkage_file": "probe_dendrogram_linkage.csv",
        }

    return {
        "experiment": "Exp018 — Universal Safety Representation (cross-benchmark generalization)",
        "hypothesis": "Do multiple safety benchmarks share a benchmark-independent activation-space representation of unsafe intent?",
        "model": CONFIG.model_name,
        "layers": list(CONFIG.layers),
        "benchmarks_enabled": list(CONFIG.benchmarks_enabled),
        "benchmark_load_status": BENCHMARK_LOAD_STATUS,
        "dataset_sizes": {name: len(df) for name, df in BENCHMARK_DFS.items()},
        "prompt_token_stats": PROMPT_LENGTH_DF.to_dict("records") if len(PROMPT_LENGTH_DF) else [],
        "max_prompt_tokens": CONFIG.max_prompt_tokens,
        "max_examples_per_benchmark": CONFIG.max_examples_per_benchmark,
        "best_train_configuration": best_row,
        "largest_transfer_gap": largest_transfer_gap,
        "transfer_matrix_summary": transfer_matrix_summary,
        "lobo_summary_secondary": lobo_summary,
        "probe_similarity_stats": {
            "mean_cosine_similarity": float(PROBE_SIM_DF["cosine_similarity"].mean()) if len(PROBE_SIM_DF) else None,
            "min_cosine_similarity": float(PROBE_SIM_DF["cosine_similarity"].min()) if len(PROBE_SIM_DF) else None,
            "max_cosine_similarity": float(PROBE_SIM_DF["cosine_similarity"].max()) if len(PROBE_SIM_DF) else None,
        },
        "probe_geometry_summary": probe_geometry_summary,
        "layer_stability_stats": {
            "mean_cosine_similarity": float(LAYER_SIM_DF["cosine_similarity"].mean()) if len(LAYER_SIM_DF) else None,
        },
        "seed_stability_stats": SEED_STABILITY_SUMMARY_DF.to_dict("records") if len(SEED_STABILITY_SUMMARY_DF) else [],
        "permutation_control": permutation_summary,
        "random_direction_baseline": random_baseline_summary,
        "runtime": {
            "started": REPRO.get("timestamp") if isinstance(REPRO, dict) else None,
            "completed": datetime.now().isoformat(),
        },
        "timestamp": datetime.now().isoformat(),
        "configuration": asdict(CONFIG),
        "reproducibility": REPRO,
        "caveats": [
            f"{name} used a PLACEHOLDER, not a real loaded dataset" for name, ok in BENCHMARK_LOAD_STATUS.items() if not ok
        ],
    }

SUMMARY = build_summary()
with open(DIRS["root"] / "exp018_summary.json", "w") as f:
    json.dump(SUMMARY, f, indent=2, default=str)

print(json.dumps(SUMMARY, indent=2, default=str))
print(f"\n[Exp018] Notebook run complete. All outputs saved under {DIRS['root']}")
